In [ ]:
# =============================================================================
# Experiment 6: Detecting Abnormal URL Patterns using Feature Engineering
# Objective: Using a set of URLs, detect abnormal patterns using feature engineering. [CO3]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import math
from collections import Counter
from urllib.parse import urlparse, parse_qs
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Create Dataset of Normal and Abnormal/Obfuscated URLs
# =============================================================================

urls = [
    # Normal, clean URLs
    'https://www.google.com/search?q=machine+learning',
    'https://github.com/scikit-learn/scikit-learn',
    'https://docs.python.org/3/library/urllib.parse.html',
    'https://www.wikipedia.org/wiki/Cybersecurity',
    'https://stackoverflow.com/questions/123456/python-pandas',
    'https://www.jainuniversity.ac.in/academics',
    'https://www.cloudflare.com/learning/ddos/what-is-a-ddos-attack/',
    'https://www.amazon.in/dp/B09G9FPGTN',
    'https://www.linkedin.com/in/ajay-bose/',
    'https://www.microsoft.com/en-in/security',
    'https://www.kaggle.com/competitions',
    'https://fastapi.tiangolo.com/tutorial/',
    'https://www.postgresql.org/docs/current/',
    'https://reactjs.org/docs/getting-started.html',
    'https://www.bbc.com/news/technology',
    # Abnormal / Obfuscated / Phishing / Payload URLs
    'http://192.168.10.55:8080/admin/../../etc/passwd',
    'http://login.verify-account.secure-update.bank-auth-portal.xyz/signin?token=a8f9d8s7f6a8s7df6a8s7df68a7sdf68',
    'http://paypal.com@evil-phishing-server.ru/login.php?redirect=http://malicious.site/payload.exe',
    'http://bit.ly/3x99ZkL?cmd=powershell%20-enc%20SQBFAFgAIAAoAE4AZQB3AC0ATwBiAGoAZQBjAHQ=',
    'http://0x7f000001/shell.php?id=1%20UNION%20SELECT%201,2,3,4--'
]

df = pd.DataFrame({'URL': urls})
print(f"Total URLs loaded: {len(df)}")
print(df.head())

In [ ]:
# =============================================================================
# 3. Advanced Feature Engineering for Abnormal URL Detection
# Engineering lexical, structural, obfuscation, and entropy-based features.
# =============================================================================

def calculate_shannon_entropy(text):
    """Measures randomness in a string; high entropy often indicates randomized domains or encoded payloads."""
    if not text:
        return 0.0
    counts = Counter(text)
    length = len(text)
    return -sum((count / length) * math.log2(count / length) for count in counts.values())

def engineer_url_features(url):
    parsed = urlparse(url)
    hostname = parsed.netloc
    path = parsed.path
    query = parsed.query
    
    # 1. Length-based features
    url_len = len(url)
    host_len = len(hostname)
    path_len = len(path)
    query_len = len(query)
    
    # 2. Structural & Subdomain depth features
    subdomain_count = max(0, hostname.count('.') - 1)
    path_depth = path.count('/')
    query_param_count = len(parse_qs(query))
    
    # 3. Character distribution & ratios
    digit_ratio = sum(c.isdigit() for c in url) / max(1, url_len)
    special_char_count = len(re.findall(r'[^a-zA-Z0-9]', url))
    special_char_ratio = special_char_count / max(1, url_len)
    
    # 4. Obfuscation & Attack indicators
    has_at_symbol = 1 if '@' in url else 0  # Credential/redirect masking
    has_hex_or_encoding = 1 if ('%' in url or '0x' in hostname.lower()) else 0
    has_ip_or_port = 1 if re.search(r'(\d+\.\d+\.\d+\.\d+|:\d+)', hostname) else 0
    has_traversal_or_sqli = 1 if re.search(r'(\.\./|union|select|cmd=|powershell|\.exe|\.php)', url.lower()) else 0
    
    # 5. Information Entropy
    url_entropy = calculate_shannon_entropy(url)
    
    return pd.Series({
        'url_len': url_len,
        'host_len': host_len,
        'path_len': path_len,
        'query_len': query_len,
        'subdomain_count': subdomain_count,
        'path_depth': path_depth,
        'query_param_count': query_param_count,
        'digit_ratio': round(digit_ratio, 4),
        'special_char_ratio': round(special_char_ratio, 4),
        'has_at_symbol': has_at_symbol,
        'has_hex_or_encoding': has_hex_or_encoding,
        'has_ip_or_port': has_ip_or_port,
        'has_traversal_or_sqli': has_traversal_or_sqli,
        'url_entropy': round(url_entropy, 4)
    })

features_df = df['URL'].apply(engineer_url_features)
print("--- Engineered Features Summary (First 5 Rows) ---")
print(features_df.head())

In [ ]:
# =============================================================================
# 4. Unsupervised Abnormal Pattern Detection (Isolation Forest)
# Using engineered features to automatically isolate anomalous URLs.
# =============================================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_df)

# Fit Isolation Forest (contamination=0.25 reflects expected proportion of abnormal URLs)
iso_forest = IsolationForest(n_estimators=100, contamination=0.25, random_state=42)
df['Anomaly_Score'] = iso_forest.fit_predict(X_scaled)
df['Decision_Score'] = iso_forest.decision_function(X_scaled)
df['Pattern_Status'] = df['Anomaly_Score'].map({1: 'Normal', -1: 'Abnormal'})

print("--- Detected Abnormal URLs ---")
for idx, row in df[df['Pattern_Status'] == 'Abnormal'].iterrows():
    print(f"[ABNORMAL | Score: {row['Decision_Score']:.3f}] {row['URL']}")

In [ ]:
# =============================================================================
# 5. Visualize Engineered Feature Distributions & Anomaly Separation
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: URL Length vs Shannon Entropy colored by Pattern Status
plot_df = pd.concat([df[['Pattern_Status']], features_df], axis=1)
sns.scatterplot(
    data=plot_df, x='url_len', y='url_entropy', hue='Pattern_Status',
    palette={'Normal': 'steelblue', 'Abnormal': 'crimson'}, s=120, ax=axes[0]
)
axes[0].set_title('URL Length vs. Shannon Entropy')
axes[0].set_xlabel('URL Length (characters)')
axes[0].set_ylabel('Shannon Entropy')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Plot 2: 2D PCA Projection of all 14 Engineered Features
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)
plot_df['PCA1'] = pca_coords[:, 0]
plot_df['PCA2'] = pca_coords[:, 1]

sns.scatterplot(
    data=plot_df, x='PCA1', y='PCA2', hue='Pattern_Status',
    palette={'Normal': 'steelblue', 'Abnormal': 'crimson'}, s=120, ax=axes[1]
)
axes[1].set_title('PCA Projection of Engineered URL Features')
axes[1].set_xlabel('Principal Component 1')
axes[1].set_ylabel('Principal Component 2')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()